<a href="https://colab.research.google.com/github/ubaidullah7598-alt/Deep-Research-Ai-agent/blob/main/quickstarts/Get_started_managed_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Mount Google Drive and Extract Dataset
We will mount Google Drive to ensure we can save the model weights and results. Then we will unzip `/content/files (2).zip` to explore its contents.

In [4]:
from google.colab import drive
import zipfile
import os

# Try to mount Google Drive, fallback to local storage if it fails
try:
    print("Attempting to mount Google Drive...")
    drive.mount('/content/drive', timeout_ms=30000)
    save_dir = '/content/drive/MyDrive/training_results'
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"Could not mount Google Drive due to: {e}")
    print("Falling back to local directory for saving results.")
    save_dir = '/content/training_results'

os.makedirs(save_dir, exist_ok=True)
print(f"Results and training progress will be saved to: {save_dir}")

# Define paths
zip_path = '/content/files (2).zip'
extract_path = '/content/dataset'

# Unzip the file if it exists
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Extracted zip to {extract_path}")
    if os.path.exists(extract_path):
        print("Contents:", os.listdir(extract_path))
else:
    print(f"Zip file not found at {zip_path}. Please make sure the zip file is uploaded.")

Attempting to mount Google Drive...
Could not mount Google Drive due to: Error: credential propagation was unsuccessful
Falling back to local directory for saving results.
Results and training progress will be saved to: /content/training_results
Extracted zip to /content/dataset
Contents: ['DunbaaBERT_Project.zip', 'dunbaa_training.ipynb']


# Step 2: Load and Inspect CSV Files
Now we will load `/content/train (1).csv` and `/content/validation.csv` using pandas to inspect their structures.

In [2]:
import pandas as pd

train_df = pd.read_csv('/content/train (1).csv')
val_df = pd.read_csv('/content/validation.csv')

print("Train dataset shape:", train_df.shape)
print("Validation dataset shape:", val_df.shape)

print("\nTrain columns and types:")
display(train_df.info())

print("\nFirst 5 rows of Training Data:")
display(train_df.head())

Train dataset shape: (65499, 7)
Validation dataset shape: (8207, 7)

Train columns and types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65499 entries, 0 to 65498
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sample_id  65499 non-null  object
 1   text       65499 non-null  object
 2   script     65499 non-null  object
 3   label      65499 non-null  object
 4   dataset    65499 non-null  object
 5   group_id   65499 non-null  object
 6   split      65499 non-null  object
dtypes: object(7)
memory usage: 3.5+ MB


None


First 5 rows of Training Data:


,sample_id,text,script,label,dataset,group_id,split
0,URDU_000001,لاہور (ڈیلی پاکستان آن لائن) وفاقی حکومت نے غی...,Urdu,real,UrduFake,G000001,train
1,URDU_000002,دادو (ویب ڈیسک ) سوات میں سخت ٹھنڈ اور شدید بر...,Urdu,real,UrduFake,G000002,train
2,URDU_000003,cnnurdu نے tuesday، 22 may 2018 کو شائع کیا. م...,Urdu,real,UrduFake,G000003,train
3,URDU_000004,دبئی 24 نومبر (سیاست ڈاٹ کام ) انٹرنیشنل کرکٹ ...,Urdu,real,UrduFake,G000004,train
4,URDU_000005,چین کے صدر شی جن پنگ نے منگل کے روز چین کی منڈ...,Urdu,real,UrduFake,G000005,train


# Step 3: Train and Evaluate Model
We will use a TF-IDF vectorizer and a Logistic Regression model to classify the text into 'real' or 'fake'. All model weights and training reports will be saved to our configured directory.

In [5]:
import os
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Ensure we have our target labels as categories
y_train = train_df['label']
y_val = val_df['label']

# Extract text features
print("Vectorizing text data...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])

# Initialize and train the model
print("Training the classifier...")
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Evaluate the model
print("Evaluating the model...")
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
report = classification_report(y_val, y_pred)

print(f"Validation Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", report)

# Define the destination directory (checking save_dir defined in Step 1, falling back if not defined)
try:
    save_path = save_dir
except NameError:
    save_path = '/content/training_results'
os.makedirs(save_path, exist_ok=True)

# Save model, vectorizer, and results
model_file = os.path.join(save_path, 'model.pkl')
vectorizer_file = os.path.join(save_path, 'vectorizer.pkl')
report_file = os.path.join(save_path, 'results.txt')

with open(model_file, 'wb') as f:
    pickle.dump(model, f)
with open(vectorizer_file, 'wb') as f:
    pickle.dump(vectorizer, f)
with open(report_file, 'w') as f:
    f.write(f"Accuracy: {accuracy:.4f}\n\nClassification Report:\n{report}")

print(f"Training successfully finished! Model, vectorizer, and metrics are saved under: {save_path}")

Vectorizing text data...
Training the classifier...
Evaluating the model...
Validation Accuracy: 0.8906

Classification Report:
               precision    recall  f1-score   support

        fake       0.89      0.87      0.88      3840
        real       0.89      0.91      0.90      4367

    accuracy                           0.89      8207
   macro avg       0.89      0.89      0.89      8207
weighted avg       0.89      0.89      0.89      8207

Training successfully finished! Model, vectorizer, and metrics are saved under: /content/training_results


# Step 4: Extract and Inspect the Inner Project Zip
We will extract `DunbaaBERT_Project.zip` to find the main code files, training scripts, or configurations.

In [6]:
import zipfile
import os

inner_zip = '/content/dataset/DunbaaBERT_Project.zip'
project_dir = '/content/dunbaa_project'

if os.path.exists(inner_zip):
    with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
        zip_ref.extractall(project_dir)
    print(f"Extracted inner project zip to {project_dir}")

    # List all extracted items recursively to find execution scripts
    for root, dirs, files in os.walk(project_dir):
        level = root.replace(project_dir, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
else:
    print(f"Inner zip file not found at {inner_zip}")

Extracted inner project zip to /content/dunbaa_project
dunbaa_project/
    DunbaaBERT_Project/
        README.md
        requirements.txt
        notebooks/
            01_data_audit_eda.ipynb
            04_training.ipynb
            06_evals_calibration_robustness.ipynb
            05_evaluation.ipynb
            07_visualization.ipynb
            00_setup_colab.ipynb
            03_model_setup.ipynb
            02_tokenization.ipynb
            dunbaa_training.ipynb
            08_report_export.ipynb
        configs/
            dunbaa_default.json
        outputs/
            data_audit/
                eda_lengths.png
                eda_class_balance.png
                eda_composition.png
                data_audit.json
        src/
            dunbaa/
                visualize.py
                tokenization.py
                model.py
                train.py
                evals.py
                utils.py
                config.py
                evaluate.py
               

# Step 5: Setup Project Data and Install Requirements
We will copy the CSV dataset files (`train (1).csv` and `validation.csv`) into the project's data directory so that the pipeline script can find them. We will also install any libraries listed in `requirements.txt`.

In [7]:
import shutil
import os

# Source paths
train_src = '/content/train (1).csv'
val_src = '/content/validation.csv'

# Destination paths
project_data_dir = '/content/dunbaa_project/DunbaaBERT_Project/data'
os.makedirs(project_data_dir, exist_ok=True)

train_dst = os.path.join(project_data_dir, 'train.csv')
val_dst = os.path.join(project_data_dir, 'validation.csv')

# Copy CSVs into place
shutil.copy(train_src, train_dst)
shutil.copy(val_src, val_dst)

print("Dataset copied to project data folder:")
print(os.listdir(project_data_dir))

# Install requirements
requirements_path = '/content/dunbaa_project/DunbaaBERT_Project/requirements.txt'
if os.path.exists(requirements_path):
    print("Installing requirements...")
    !pip install -r {requirements_path}
else:
    print("No requirements.txt found. Using default environment.")

Dataset copied to project data folder:
['PUT_CSVS_HERE.txt', 'train.csv', 'validation.csv']
Installing requirements...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.0 MB/s  0:00:00


# Step 6: Execute the Project Pipeline
Now we will execute the provided pipeline script `run_pipeline.py`. To ensure the outputs and logs are securely backed up, we will copy the completed outputs to our configured save directory.

In [14]:
import sys
import os
import shutil
import json
import zipfile

# 1. Restore train.py to its original state from the zip
inner_zip = '/content/dataset/DunbaaBERT_Project.zip'
with zipfile.ZipFile(inner_zip, 'r') as zip_ref:
    # Extract only the original train.py file
    zip_ref.extract('DunbaaBERT_Project/src/dunbaa/train.py', '/content/dunbaa_project')

print("Restored original train.py")

# 2. Configure paths locally in the default configuration file
config_path = '/content/dunbaa_project/DunbaaBERT_Project/configs/dunbaa_default.json'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config_data = json.load(f)

    # Update directories to point to our local dataset extraction folders
    config_data['data_dir'] = '/content/dunbaa_project/DunbaaBERT_Project/data'
    config_data['work_dir'] = '/content/dunbaa_project/DunbaaBERT_Project/runs'
    config_data['deliverables_dir'] = '/content/dunbaa_project/DunbaaBERT_Project/deliverables'
    config_data['output_dir'] = '/content/dunbaa_project/DunbaaBERT_Project/outputs'

    # Remove warmup_ratio from configuration dictionary if present
    if 'warmup_ratio' in config_data:
        del config_data['warmup_ratio']

    with open(config_path, 'w') as f:
        json.dump(config_data, f, indent=4)
    print("Configuration updated successfully!")

# 3. Cleanly patch train.py to pop 'warmup_ratio' from kw right before calling TrainingArguments
train_script_path = '/content/dunbaa_project/DunbaaBERT_Project/src/dunbaa/train.py'
if os.path.exists(train_script_path):
    with open(train_script_path, 'r') as f:
        code_content = f.read()

    # Target the TrainingArguments call inside build_training_args
    target_str = "return TrainingArguments(**kw)"
    replacement_str = "kw.pop('warmup_ratio', None)\n    return TrainingArguments(**kw)"

    if target_str in code_content:
        code_content = code_content.replace(target_str, replacement_str)
        with open(train_script_path, 'w') as f:
            f.write(code_content)
        print("Successfully patched train.py to pop warmup_ratio from kw dictionary.")
    else:
        print("Target string not found in train.py.")

# Add the source directory to Python path
sys.path.append('/content/dunbaa_project/DunbaaBERT_Project/src')

# Navigate to the project root
%cd /content/dunbaa_project/DunbaaBERT_Project

try:
    print("Starting pipeline execution...")
    # Run the official pipeline entrypoint
    !python scripts/run_pipeline.py --data_dir /content/dunbaa_project/DunbaaBERT_Project/data --work_dir /content/dunbaa_project/DunbaaBERT_Project/runs --deliverables_dir /content/dunbaa_project/DunbaaBERT_Project/deliverables

    # Copy final outputs to the local training_results backup directory
    project_outputs = '/content/dunbaa_project/DunbaaBERT_Project/outputs'
    if os.path.exists(project_outputs):
        try:
            save_path = save_dir
        except NameError:
            save_path = '/content/training_results'
        shutil.copytree(project_outputs, os.path.join(save_path, 'outputs'), dirs_exist_ok=True)
        print(f"All pipeline execution results successfully backed up to {save_path}")
except Exception as e:
    print(f"An error occurred during pipeline execution: {e}")

# Return to content directory
%cd /content

Restored original train.py
Configuration updated successfully!
Successfully patched train.py to pop warmup_ratio from kw dictionary.
/content/dunbaa_project/DunbaaBERT_Project
Starting pipeline execution...
12:19:03 | INFO | train rows 65,499 | val rows 8,207 | leakage audit {'sample_id_overlap': 0, 'group_id_overlap': 0, 'val_rows_with_text_also_in_train': 0, 'val_unique_texts_also_in_train': 0}
Loading weights: 100% 197/197 [00:00<00:00, 4138.65it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: DunbaaBERT/DunbaaBERT_32k_base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_p